# 2. Train and evaluate TFT using the prepared Snowflake export

**ICL remains 365 and LSTM layers remain 4.** No local spreadsheet input is used.
The default is a 2,000-series pilot, training through August 31, 2025 and scoring
September 1–December 7, 2025. This is a festive backtest, not a forecast for 2026.
Set MAX_SERIES=None for the full population after the pilot. Results cover the
selected population; they are not automatically Runner-only unless the source was filtered.

For a feature comparison, prepare legacy and engineered Snowflake exports from the
same underlying sales snapshot. Run this notebook once per ROLES_PATH. Keep every
training setting identical. The optional comparison cell checks population, target,
weights and settings before reporting differences. Different feature counts can
change convergence; report actual update counts as well as the common budget.

For final 2026 production training, set RUN_MODE="production", select the feature
export, set MAX_SERIES=None and explicitly set PRODUCTION_EPOCHS from your backtest
experiment. It trains through August 31, 2026 without an artificial in-sample validation
score. Train for a chosen budget; no accuracy guarantee follows from more epochs.

Cache artifacts, fitted encoding, shared covariates, model, settings, predictions and
metrics are saved together for this run. Do not point the old inference notebook at
this new cache layout; this notebook includes compatible inference.

In [ ]:
import os, json, gc, hashlib, pickle, time, collections.abc
from pathlib import Path
from datetime import datetime
from uuid import uuid4
import numpy as np
import pandas as pd
import torch
import darts
from darts import TimeSeries
from darts.models import TFTModel
from darts.utils.likelihood_models import NegativeBinomialLikelihood
from pytorch_lightning.callbacks import EarlyStopping, Callback
from sklearn.preprocessing import OrdinalEncoder
from scipy import stats

BASE=Path.cwd()
ROLES_PATH=None  # Or an explicit column_roles.json from a previous export.
if ROLES_PATH is None:
    ROLES_PATH=json.loads((BASE/'active_tft_export.json').read_text())['roles_path']
ROLES_PATH=Path(ROLES_PATH).resolve()
ROLES=json.loads(ROLES_PATH.read_text());DATA_DIR=ROLES_PATH.parent
if ROLES.get('schema_version')!=2:raise ValueError('Run the revised chunking notebook first.')
RUN_MODE='backtest'                 # 'backtest' or 'production'
MAX_SERIES=2000                     # None = full population. Same value for both feature sets.
INPUT_CHUNK_LENGTH=365
OUTPUT_CHUNK_LENGTH=98
LSTM_LAYERS=4
HIDDEN_SIZE=32
BATCH_SIZE=256                      # If GPU OOM, use 128 in BOTH comparisons and restart.
MAX_EPOCHS=20
LIMIT_TRAIN_BATCHES=250             # Pilot budget. Increase only after checking time/learning curves.
PRODUCTION_EPOCHS=None              # Required for production; choose from completed experiments.
FESTIVE_MULTIPLIER=4.0              # Explicit 4x, matching the previous 1+3 calculation.
SEED=42
PREDICT_BLOCK=256
QUANTILES=[50,60,70,72,80]
NUM_WORKERS=0                      # Safe for Windows/Jupyter; benchmark workers separately.
CACHE_SERIES_IN_RAM=True

if RUN_MODE not in {'backtest','production'}:raise ValueError('Unknown RUN_MODE')
if INPUT_CHUNK_LENGTH!=365:raise ValueError('This experiment preserves ICL=365.')
if RUN_MODE=='production' and (not isinstance(PRODUCTION_EPOCHS,int) or PRODUCTION_EPOCHS<1):
    raise ValueError('Set PRODUCTION_EPOCHS explicitly from the experiment results.')
TRAIN_START=pd.Timestamp(ROLES['history_start'])
TRAIN_END=pd.Timestamp('2025-08-31' if RUN_MODE=='backtest' else '2026-08-31')
PRED_START=pd.Timestamp('2025-09-01' if RUN_MODE=='backtest' else '2026-09-01')
PRED_END=PRED_START+pd.Timedelta(days=OUTPUT_CHUNK_LENGTH-1)
WARMUP_START=PRED_START-pd.Timedelta(days=INPUT_CHUNK_LENGTH)
MIN_LEN=INPUT_CHUNK_LENGTH+OUTPUT_CHUNK_LENGTH
time_col,group_col,target_col=[ROLES[k] for k in ['time_col','group_col','target_col']]
static_covariates=ROLES['static_covariates'];future_covariates=ROLES['future_covariates']
FREQ=ROLES['freq']
if PRED_END>pd.Timestamp(ROLES['calendar_end']):raise ValueError('Calendar does not cover forecast.')
if TRAIN_END>pd.Timestamp(ROLES['history_end']):raise ValueError('History does not cover training.')
if RUN_MODE=='backtest' and PRED_END>pd.Timestamp(ROLES['history_end']):raise ValueError('Backtest actuals are unavailable.')
if not torch.cuda.is_available():raise RuntimeError('CUDA is unavailable. Select your GPU-enabled Python kernel.')
PRECISION='bf16-mixed' if torch.cuda.is_bf16_supported() else '32-true'
torch.set_float32_matmul_precision('high')
torch.backends.cudnn.benchmark=True
device=torch.cuda.get_device_properties(0)
print('Darts:',darts.__version__,'Torch:',torch.__version__)
print('GPU:',device.name,'VRAM GiB:',round(device.total_memory/2**30,1),'Precision:',PRECISION)
print('Train:',TRAIN_START.date(),TRAIN_END.date(),'Score/forecast:',PRED_START.date(),PRED_END.date())
print('ICL:',INPUT_CHUNK_LENGTH,'LSTM layers:',LSTM_LAYERS)

def digest(obj):return hashlib.sha256(json.dumps(obj,sort_keys=True,default=str).encode()).hexdigest()
def file_hash(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()
def atomic_json(path,obj):
    path=Path(path);tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(obj,indent=2,default=str));os.replace(tmp,path)
def safe_name(key):return hashlib.sha256(str(key).encode()).hexdigest()

In [ ]:
# Explicit common weighting window: N-16..N+10 OR D-3..D+6.
# This remains identical between legacy and engineered feature variants.
# It does not treat every nonzero continuous festival distance as a festive day.
def make_weights(dates,anchors,multiplier):
    if multiplier<1:raise ValueError('Festive multiplier must be at least 1.')
    dates=pd.DatetimeIndex(dates);mask=np.zeros(len(dates),dtype=bool)
    for a in anchors:
        n=pd.Timestamp(a['navratri_start']);d=pd.Timestamp(a['diwali'])
        mask |= ((dates>=n-pd.Timedelta(days=16)) & (dates<=n+pd.Timedelta(days=10)))
        mask |= ((dates>=d-pd.Timedelta(days=3)) & (dates<=d+pd.Timedelta(days=6)))
    return np.where(mask,multiplier,1.0).astype('float32')

def choose_series(index,maximum):
    if index[group_col].duplicated().any():raise ValueError('Duplicate series in index.')
    out=index.copy()
    out['_hash']=out[group_col].map(lambda k:hashlib.sha256(f'{SEED}|{k}'.encode()).hexdigest())
    strata=['ZONAL_OFFICE_NAME','MODEL_FAMILY']
    out=out.sort_values('_hash')
    out['_rank']=out.groupby(strata,dropna=False).cumcount()
    out=out.sort_values(['_rank','_hash'])
    if maximum is not None:
        if maximum<1:raise ValueError('MAX_SERIES must be positive or None.')
        out=out.head(maximum)
    return out.drop(columns=['_hash','_rank']).sort_values(group_col).reset_index(drop=True)

def split_history(g):
    g=g.sort_values(time_col).copy();dates=pd.DatetimeIndex(pd.to_datetime(g[time_col]))
    if dates.has_duplicates or not dates.equals(pd.date_range(dates.min(),dates.max(),freq=FREQ)):
        raise ValueError('Missing/duplicate dates; the source must explicitly contain zero-sales days.')
    y=g[target_col].to_numpy(dtype=np.float32)
    if not np.isfinite(y).all() or (y<0).any() or not np.equal(y,np.rint(y)).all():
        raise ValueError('Target is not a non-negative integer count series.')
    train=(dates>=TRAIN_START)&(dates<=TRAIN_END)
    hist=(dates>=WARMUP_START)&(dates<PRED_START)
    if train.sum()<MIN_LEN or hist.sum()!=INPUT_CHUNK_LENGTH or dates[train][-1]!=TRAIN_END:
        return None
    if y[train].sum()==0:return None  # Eligibility uses training observations only.
    w=make_weights(dates,ROLES['anchors'],FESTIVE_MULTIPLIER)
    result={'train_sales':y[train],'train_weight':w[train],
            'train_start':np.array(str(dates[train][0].date())),
            'history_sales':y[hist],'history_start':np.array(str(WARMUP_START.date()))}
    if RUN_MODE=='backtest':
        val=(dates>=WARMUP_START)&(dates<=PRED_END)
        if val.sum()!=MIN_LEN:return None
        result.update(val_sales=y[val],val_weight=w[val],val_start=np.array(str(WARMUP_START.date())))
    return result

for name,key in [('shared_calendar.parquet','calendar_sha256'),('series_index.parquet','index_sha256')]:
    if file_hash(DATA_DIR/name)!=ROLES[key]:raise ValueError(f'{name} changed after extraction.')
cal=pd.read_parquet(DATA_DIR/'shared_calendar.parquet')
cal[time_col]=pd.to_datetime(cal[time_col])
if cal[time_col].duplicated().any() or not np.isfinite(cal[future_covariates].to_numpy()).all():
    raise ValueError('Invalid shared calendar.')
SHARED_COV=TimeSeries.from_dataframe(cal,time_col=time_col,value_cols=future_covariates,freq=FREQ).astype(np.float32)
index=choose_series(pd.read_parquet(DATA_DIR/'series_index.parquet'),MAX_SERIES)
cache_config={'version':2,'export_id':ROLES['export_id'],'chunks':ROLES['chunks'],
              'train_start':TRAIN_START,'train_end':TRAIN_END,'pred_start':PRED_START,'pred_end':PRED_END,
              'icl':INPUT_CHUNK_LENGTH,'ocl':OUTPUT_CHUNK_LENGTH,'mode':RUN_MODE,
              'multiplier':FESTIVE_MULTIPLIER,'anchors':ROLES['anchors'],'keys':index[group_col].tolist()}
CACHE_DIR=BASE/'tft_series_cache'/digest(cache_config)[:20]
CACHE_DIR.mkdir(parents=True,exist_ok=True)
manifest_path=CACHE_DIR/'manifest.json'
if manifest_path.exists():
    manifest=json.loads(manifest_path.read_text())
    if manifest['config_hash']!=digest(cache_config):raise ValueError('Cache configuration mismatch.')
    # Verify source byte hashes before reusing a cache, not merely file existence.
    for item in ROLES['chunks']:
        if item['name'] in set(index.CHUNK_FILE) and file_hash(DATA_DIR/'history'/item['name'])!=item['sha256']:
            raise ValueError('Source history changed. Re-extract it before training.')
    accepted=manifest['series_keys']
    if any(not (CACHE_DIR/(safe_name(k)+'.npz')).exists() for k in accepted):
        raise ValueError('Incomplete series cache; remove its manifest to rebuild.')
else:
    accepted=[];chunk_info={x['name']:x for x in ROLES['chunks']}
    for name,keys_frame in index.groupby('CHUNK_FILE',sort=True):
        path=DATA_DIR/'history'/name
        if file_hash(path)!=chunk_info[name]['sha256']:raise ValueError(f'History changed: {name}')
        frame=pd.read_parquet(path,columns=[group_col,time_col,target_col],
                              filters=[(group_col,'in',keys_frame[group_col].tolist())])
        frame[time_col]=pd.to_datetime(frame[time_col])
        for key,g in frame.groupby(group_col,sort=True):
            payload=split_history(g)
            if payload is None:continue
            path_out=CACHE_DIR/(safe_name(key)+'.npz')
            with open(path_out.with_suffix('.tmp'),'wb') as f:np.savez(f,**payload)
            os.replace(path_out.with_suffix('.tmp'),path_out);accepted.append(str(key))
        del frame;gc.collect()
        print('Cached',len(accepted),'eligible series')
    accepted=sorted(accepted)
    manifest={'config_hash':digest(cache_config),'series_keys':accepted}
    atomic_json(manifest_path,manifest)
index=index.set_index(group_col).loc[accepted].reset_index()
if len(index)==0:raise ValueError('No eligible series for this split.')
series_keys=index[group_col].tolist()
print(f'Using {len(series_keys):,} series; {len(cache_config["keys"])-len(series_keys):,} skipped for insufficient/zero training history.')

In [ ]:
# Encode static values once in a vectorized operation. All selected series are
# training series; the encoder never fits on held-out target values.
static_raw=index[static_covariates].astype(str)
encoder=OrdinalEncoder(handle_unknown='error',dtype=np.float32)
encoded=encoder.fit_transform(static_raw)
CATEGORY_COUNTS={c:len(encoder.categories_[i]) for i,c in enumerate(static_covariates)}
STATIC_ENCODED=[pd.DataFrame(encoded[i:i+1],columns=static_covariates) for i in range(len(index))]

class TargetSequence(collections.abc.Sequence):
    def __init__(self,split):
        self.split=split;self.ram={}
    def __len__(self):return len(series_keys)
    def __getitem__(self,i):
        if isinstance(i,slice):return [self[j] for j in range(*i.indices(len(self)))]
        if i<0:i+=len(self)
        if not 0<=i<len(self):raise IndexError(i)
        if i in self.ram:return self.ram[i]
        with np.load(CACHE_DIR/(safe_name(series_keys[i])+'.npz'),allow_pickle=False) as z:
            vals=z[self.split+'_sales'];start=str(z[self.split+'_start'])
        ts=TimeSeries.from_times_and_values(pd.date_range(start,periods=len(vals),freq=FREQ),
            vals.reshape(-1,1),columns=[target_col],static_covariates=STATIC_ENCODED[i])
        if CACHE_SERIES_IN_RAM:self.ram[i]=ts
        return ts

class WeightSequence(collections.abc.Sequence):
    def __init__(self,split):self.split=split;self.ram={}
    def __len__(self):return len(series_keys)
    def __getitem__(self,i):
        if isinstance(i,slice):return [self[j] for j in range(*i.indices(len(self)))]
        if i<0:i+=len(self)
        if not 0<=i<len(self):raise IndexError(i)
        if i in self.ram:return self.ram[i]
        with np.load(CACHE_DIR/(safe_name(series_keys[i])+'.npz'),allow_pickle=False) as z:
            w=z[self.split+'_weight'];start=str(z[self.split+'_start'])
        ts=TimeSeries.from_times_and_values(pd.date_range(start,periods=len(w),freq=FREQ),w[:,None],columns=['WEIGHT'])
        if CACHE_SERIES_IN_RAM:self.ram[i]=ts
        return ts

train_seq=TargetSequence('train');history_seq=TargetSequence('history')
train_weights=WeightSequence('train')
val_seq=TargetSequence('val') if RUN_MODE=='backtest' else None
val_weights=WeightSequence('val') if RUN_MODE=='backtest' else None
# Materialize the pilot as a plain list so Darts sees static column names directly.
# Training/validation lists remain in host RAM even with caching disabled.
# Check available RAM before setting MAX_SERIES=None; use the pilot to measure it.
train_data=list(train_seq)
val_data=list(val_seq) if val_seq is not None else None
if not train_data[0].static_covariates.columns.equals(pd.Index(static_covariates)):
    raise ValueError('Static column order changed during construction.')
all_cov=[SHARED_COV]*len(series_keys)  # References, not copies.

RUN_ID=datetime.now().strftime('%Y%m%d_%H%M%S')+'_'+uuid4().hex[:8]
MODEL_NAME=f'tft_{ROLES["feature_set"]}_{RUN_MODE}_{RUN_ID}'
RUN_DIR=BASE/'tft_runs'/MODEL_NAME;RUN_DIR.mkdir(parents=True,exist_ok=False)
MODEL_WORK_DIR=BASE/'tft_models'
SHARED_COV.to_pickle(RUN_DIR/'shared_cov.pkl')
with open(RUN_DIR/'static_encoder.pkl','wb') as f:pickle.dump(encoder,f)
index.to_parquet(RUN_DIR/'series_index.parquet',index=False)
atomic_json(RUN_DIR/'source_roles.json',ROLES)

# Hash the actual training labels and dates, not just source filenames, for A/B checks.
target_hash=hashlib.sha256()
for key,ts in zip(series_keys,train_data):
    target_hash.update(key.encode());target_hash.update(ts.time_index.asi8.tobytes())
    target_hash.update(ts.values(copy=False).tobytes())
comparison_config={'mode':RUN_MODE,'train_start':str(TRAIN_START.date()),'train_end':str(TRAIN_END.date()),
    'pred_start':str(PRED_START.date()),'pred_end':str(PRED_END.date()),'icl':INPUT_CHUNK_LENGTH,
    'ocl':OUTPUT_CHUNK_LENGTH,'layers':LSTM_LAYERS,'hidden_size':HIDDEN_SIZE,'dropout':0.05,
    'batch_size':BATCH_SIZE,'max_epochs':MAX_EPOCHS if RUN_MODE=='backtest' else PRODUCTION_EPOCHS,
    'limit_train_batches':LIMIT_TRAIN_BATCHES,'multiplier':FESTIVE_MULTIPLIER,'anchors':ROLES['anchors'],
    'seed':SEED,'precision':PRECISION,'keys_hash':digest(series_keys),'train_target_hash':target_hash.hexdigest(),
    'static_hash':hashlib.sha256(pd.util.hash_pandas_object(static_raw,index=False).values.tobytes()).hexdigest()}
atomic_json(RUN_DIR/'run_config.json',dict(comparison_config,source_roles_path=str(ROLES_PATH),
    feature_set=ROLES['feature_set'],cache_dir=str(CACHE_DIR),num_series=len(series_keys),future_covariates=future_covariates))

In [ ]:
class TrainingMonitor(Callback):
    def __init__(self):self.records=[]
    def on_train_start(self,trainer,pl_module):
        count=sum(p.numel() for p in pl_module.input_embeddings.parameters())
        if count==0:raise RuntimeError('No categorical embeddings were created. Check static column names and checkpoint settings.')
        print('Categorical embedding parameters:',count)
    def on_train_epoch_start(self,trainer,pl_module):
        torch.cuda.synchronize();torch.cuda.reset_peak_memory_stats()
        self.started=time.perf_counter();self.start_step=trainer.global_step
    def on_train_epoch_end(self,trainer,pl_module):
        torch.cuda.synchronize()
        record={'epoch':trainer.current_epoch+1,'updates':trainer.global_step-self.start_step,
                'global_step':trainer.global_step,'epoch_seconds':time.perf_counter()-self.started,
                'peak_gpu_GiB':torch.cuda.max_memory_allocated()/2**30}
        for name in ['train_loss','val_loss']:
            v=trainer.callback_metrics.get(name)
            record[name]=float(v.detach().cpu()) if v is not None else None
        self.records.append(record);pd.DataFrame(self.records).to_csv(RUN_DIR/'training_log.csv',index=False)
        print(record)

monitor=TrainingMonitor()
callbacks=[monitor]
if RUN_MODE=='backtest':callbacks.append(EarlyStopping(monitor='val_loss',patience=5,min_delta=1e-4,mode='min'))
embedding_sizes={c:(CATEGORY_COUNTS[c],min(50,(CATEGORY_COUNTS[c]+1)//2)) for c in static_covariates}
model=TFTModel(
    input_chunk_length=INPUT_CHUNK_LENGTH,output_chunk_length=OUTPUT_CHUNK_LENGTH,
    hidden_size=HIDDEN_SIZE,lstm_layers=LSTM_LAYERS,num_attention_heads=4,dropout=0.05,
    batch_size=BATCH_SIZE,n_epochs=MAX_EPOCHS if RUN_MODE=='backtest' else PRODUCTION_EPOCHS,
    likelihood=NegativeBinomialLikelihood(),loss_fn=None,use_reversible_instance_norm=False,
    categorical_embedding_sizes=embedding_sizes,random_state=SEED,add_relative_index=True,
    save_checkpoints=True,force_reset=False,model_name=MODEL_NAME,work_dir=str(MODEL_WORK_DIR),
    pl_trainer_kwargs={'accelerator':'gpu','devices':1,'callbacks':callbacks,'gradient_clip_val':0.1,
        'precision':PRECISION,'accumulate_grad_batches':1,'limit_train_batches':LIMIT_TRAIN_BATCHES,
        'limit_val_batches':1.0,'num_sanity_val_steps':0,'log_every_n_steps':25},
)
fit_kwargs=dict(series=train_data,future_covariates=all_cov,sample_weight=train_weights,
    dataloader_kwargs={'num_workers':NUM_WORKERS,'pin_memory':True},verbose=True)
if RUN_MODE=='backtest':
    fit_kwargs.update(val_series=val_data,val_future_covariates=all_cov,val_sample_weight=val_weights)
started=time.perf_counter()
# Fail clearly on unsupported arguments; do not silently switch to unweighted training.
model.fit(**fit_kwargs)
TRAIN_SECONDS=time.perf_counter()-started
TOTAL_UPDATES=int(model.trainer.global_step)
if RUN_MODE=='backtest':
    loaded_model=TFTModel.load_from_checkpoint(model_name=MODEL_NAME,work_dir=str(MODEL_WORK_DIR),best=True)
else:
    loaded_model=model
loaded_model.save(str(RUN_DIR/'model.pt'))
print('Saved model:',RUN_DIR/'model.pt','Training hours:',round(TRAIN_SECONDS/3600,2))

In [ ]:
# Correct closed-form inference: PyTorch's NB probability convention.
def summarise(p_ts):
    if list(p_ts.components)!=[f'{target_col}_r',f'{target_col}_p']:
        raise ValueError('Unexpected likelihood parameter names.')
    v=p_ts.values(copy=False).astype(np.float64);r,p=v[:,0],v[:,1]
    if not np.isfinite(v).all() or (r<=0).any() or (p<=0).any() or (p>=1).any():
        raise ValueError('Invalid NB parameters. Inspect output precision instead of clipping silently.')
    result={'PRED_MEAN':r*p/(1-p)}
    for q in QUANTILES:result[f'PRED_Q{q}']=stats.nbinom.ppf(q/100,r,1-p)
    if not all(np.isfinite(v).all() for v in result.values()):raise ValueError('Non-finite predictions.')
    return result

# Deterministic check against the installed Darts likelihood's own distribution.
lk=loaded_model.likelihood
raw=torch.tensor([[[[0.4,-0.2]],[[2.0,1.0]]]],dtype=torch.float64)
exported=lk.predict_likelihood_parameters(raw).detach().cpu().numpy().reshape(2,2)
dist=lk._distr_from_params(lk._params_from_output(raw))
np.testing.assert_allclose(exported[:,0]*exported[:,1]/(1-exported[:,1]),
                           dist.mean.detach().cpu().numpy().ravel(),rtol=1e-10)
pred_dir=RUN_DIR/'predictions';pred_dir.mkdir(exist_ok=True)
pred_cols=['PRED_MEAN']+[f'PRED_Q{q}' for q in QUANTILES]
totals={c:{'abs':0.0,'sq':0.0,'pred':0.0,'ape_positive':0.0} for c in pred_cols}
actual_total=0.0;n_rows=0;n_positive=0;actual_hash=hashlib.sha256();monthly_parts=[]
for lo in range(0,len(series_keys),PREDICT_BLOCK):
    hi=min(lo+PREDICT_BLOCK,len(series_keys));hist=history_seq[lo:hi]
    if any(t.end_time()!=PRED_START-pd.Timedelta(days=1) for t in hist):raise ValueError('Incorrect forecast origin.')
    params=loaded_model.predict(n=OUTPUT_CHUNK_LENGTH,series=hist,
        future_covariates=[SHARED_COV]*len(hist),predict_likelihood_parameters=True,
        num_samples=1,batch_size=BATCH_SIZE,verbose=False)
    if len(params)!=hi-lo:raise ValueError('Prediction series count mismatch.')
    rows=[]
    for offset,p_ts in enumerate(params):
        i=lo+offset;key=series_keys[i]
        if p_ts.start_time()!=PRED_START or p_ts.end_time()!=PRED_END:raise ValueError('Incorrect prediction dates.')
        row=pd.DataFrame({group_col:key,time_col:p_ts.time_index,**summarise(p_ts)})
        if RUN_MODE=='backtest':
            with np.load(CACHE_DIR/(safe_name(key)+'.npz'),allow_pickle=False) as z:y=z['val_sales'][-OUTPUT_CHUNK_LENGTH:].astype(float)
            row['ACTUAL']=y;actual_total+=y.sum();n_rows+=len(y);positive=y>0;n_positive+=positive.sum()
            actual_hash.update(key.encode());actual_hash.update(y.tobytes())
            for c in pred_cols:
                yh=row[c].to_numpy();err=yh-y
                totals[c]['abs']+=np.abs(err).sum();totals[c]['sq']+=(err**2).sum();totals[c]['pred']+=yh.sum()
                totals[c]['ape_positive']+=(np.abs(err[positive])/y[positive]).sum()
        rows.append(row)
    block=pd.concat(rows,ignore_index=True)
    block.to_parquet(pred_dir/f'block_{lo//PREDICT_BLOCK:04d}.parquet',index=False)
    monthly_parts.append(block.groupby(block[time_col].dt.to_period('M'))[pred_cols+(['ACTUAL'] if RUN_MODE=='backtest' else [])].sum())
    print('Predicted',hi,'/',len(series_keys))
    del params,rows,block;gc.collect()
monthly=pd.concat(monthly_parts).groupby(level=0).sum()
monthly.to_csv(RUN_DIR/'monthly_totals.csv');print(monthly)
if RUN_MODE=='backtest':
    if actual_total<=0:raise ValueError('Zero actual total; WAPE is undefined.')
    metrics=pd.DataFrame([{'forecast':c,'WAPE_pct':100*v['abs']/actual_total,
        'bias_pct':100*(v['pred']-actual_total)/actual_total,'MAE':v['abs']/n_rows,
        'RMSE':np.sqrt(v['sq']/n_rows),'MAPE_positive_actuals_pct':100*v['ape_positive']/n_positive if n_positive else None,
        'predicted_total':v['pred'],'actual_total':actual_total,'positive_actual_rows':int(n_positive),'rows':int(n_rows)} for c,v in totals.items()])
    metrics.to_csv(RUN_DIR/'metrics.csv',index=False);print(metrics.to_string(index=False))
    # Use the SAME preselected forecast statistic (mean by default) for the A/B decision.
    # Picking the best quantile on this backtest makes it tuning data, not an untouched test.
    summary={'feature_set':ROLES['feature_set'],'comparison_config':comparison_config,
        'actual_hash':actual_hash.hexdigest(),'metrics':metrics.to_dict('records'),
        'training_seconds':TRAIN_SECONDS,'total_updates':TOTAL_UPDATES,'run_dir':str(RUN_DIR),
        'checkpoint_selection':'minimum festive validation NLL; feature comparison uses PRED_MEAN WAPE'}
    atomic_json(RUN_DIR/'experiment_summary.json',summary)
print('Outputs:',RUN_DIR)

In [ ]:
# Optional: compare two completed backtests. Keep these None for a single run.
LEGACY_SUMMARY_PATH=None
ENGINEERED_SUMMARY_PATH=None
if LEGACY_SUMMARY_PATH and ENGINEERED_SUMMARY_PATH:
    a=json.loads(Path(LEGACY_SUMMARY_PATH).read_text());b=json.loads(Path(ENGINEERED_SUMMARY_PATH).read_text())
    if a['comparison_config']!=b['comparison_config'] or a['actual_hash']!=b['actual_hash']:
        raise ValueError('A/B runs differ in population, targets, weights or training settings. Do not attribute the difference solely to features.')
    comparison=[]
    for result in [a,b]:
        score=next(m for m in result['metrics'] if m['forecast']=='PRED_MEAN')
        comparison.append(dict(feature_set=result['feature_set'],WAPE_pct=score['WAPE_pct'],
            bias_pct=score['bias_pct'],training_hours=result['training_seconds']/3600,total_updates=result['total_updates']))
    print(pd.DataFrame(comparison).to_string(index=False))